# RQ3.1 — Input Representation

This notebook compares the information representations supplied to the LLM. Configuration columns are discovered from the current CSV, so a newly added numeric representation is included automatically.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pingouin as pg
from IPython.display import Markdown, display


SUBJECT = "project"

CONFIGURATIONS = [
    "message_only",
    "diff_only",
    "message_and_diff",
    "git_show",
]

LABELS = {
    "message_only": "Message only",
    "diff_only": "Diff only",
    "message_and_diff": "Message + diff",
    "git_show": "Git show",
}

REFERENCE_CONFIGURATION = None


# Supports running the notebook either from results/rq3_1/analysis/
# or from the repository root.
DATA_PATH = Path("../metrics/macro_f1.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("results/rq3_1/metrics/macro_f1.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find macro_f1.csv."
    )


df = pd.read_csv(DATA_PATH)

if SUBJECT not in df.columns:
    raise ValueError(
        f"Required project column '{SUBJECT}' was not found."
    )

if df[SUBJECT].duplicated().any():
    duplicates = (
        df.loc[df[SUBJECT].duplicated(), SUBJECT]
        .astype(str)
        .tolist()
    )

    raise ValueError(
        f"Duplicate project identifiers were found: {duplicates}"
    )


missing_configurations = [
    configuration
    for configuration in CONFIGURATIONS
    if configuration not in df.columns
]

if missing_configurations:
    raise ValueError(
        "Missing required input representations: "
        f"{missing_configurations}"
    )


for configuration in CONFIGURATIONS:
    numeric = pd.to_numeric(
        df[configuration],
        errors="coerce",
    )

    observed = df[configuration].notna()

    if (
        observed.any()
        and not numeric.loc[observed].notna().all()
    ):
        raise ValueError(
            f"Configuration '{configuration}' contains non-numeric values."
        )

    df[configuration] = numeric


incomplete_mask = (
    df[CONFIGURATIONS]
    .isna()
    .any(axis=1)
)

removed_projects = (
    df.loc[incomplete_mask, SUBJECT]
    .astype(str)
    .tolist()
)

df = df.loc[~incomplete_mask].copy()

if df[SUBJECT].nunique() < 3:
    raise ValueError(
        "Fewer than three complete paired projects remain."
    )


print(f"Data source: {DATA_PATH}")
print(f"Projects retained: {df[SUBJECT].nunique()}")
print(
    f"Input representations ({len(CONFIGURATIONS)}): "
    f"{CONFIGURATIONS}"
)

if removed_projects:
    print(
        "Projects removed by complete-case filtering: "
        f"{removed_projects}"
    )

display(df[[SUBJECT, *CONFIGURATIONS]].head())

Project column: project
Projects retained: 30
Detected configurations (4): ['message_only', 'diff_only', 'message_and_diff', 'git_show']


,project,message_only,diff_only,message_and_diff,git_show
0,cargo/clap,0.448229,0.559784,0.635528,0.632440
1,go/google-cloud-go,0.421964,0.369348,0.456164,0.456711
2,npm/angular,0.373868,0.318808,0.336685,0.336819
3,npm/ant-design,0.386911,0.332034,0.402581,0.428439
4,npm/biome,0.375830,0.463776,0.520424,0.464437


## Automatic statistical workflow

The project is the repeated-measures unit and project-level Macro-F1 is the dependent variable. The notebook applies the following prespecified decision rule:

1. report descriptive statistics for every detected configuration;
2. apply Shapiro–Wilk to every configuration;
3. if normality is not rejected for all configurations, use the parametric route: Mauchly's test, repeated-measures ANOVA, Greenhouse–Geisser correction when required, and Holm-corrected paired Student's $t$-tests with Hedges' $g$;
4. otherwise, use the non-parametric route: Friedman test and Holm-corrected Wilcoxon signed-rank tests with matched-pairs rank-biserial correlation ($r_{rb}$);
5. execute pairwise post-hoc tests only when the omnibus test is significant;
6. generate the results narrative from the current outputs.

All statistical decisions use unrounded values. Rounding is applied only when displaying results.


In [ ]:
from itertools import combinations
from scipy.stats import friedmanchisquare


def _format_decimal(value, digits=3):
    text = f"{float(value):.{digits}f}"
    if text.startswith("0."):
        return text[1:]
    if text.startswith("-0."):
        return "-" + text[2:]
    return text


def _format_p(p_value):
    p_value = float(p_value)
    if p_value < 0.001:
        return r"$p<.001$"
    return f"$p={_format_decimal(p_value)}$"

def _format_p_plain(p_value, digits=4):
    p_value = float(p_value)

    if p_value < 0.001:
        return "<.001"

    return f"{p_value:.{digits}f}".lstrip("0")


def _pretty_label(configuration, labels):
    if configuration in labels:
        return labels[configuration]
    return configuration.replace("_", " ").replace("-", " ").strip().title()


def _run_parametric_route(
    long_data,
    analysis_data,
    subject,
    conditions,
    dv_name,
    alpha,
):
    n_subjects = analysis_data[subject].nunique()
    k = len(conditions)

    sphericity = pg.sphericity(
        data=long_data,
        dv=dv_name,
        within="configuration",
        subject=subject,
    )
    apply_gg = bool(sphericity.pval < alpha)

    anova = pg.rm_anova(
        data=long_data,
        dv=dv_name,
        within="configuration",
        subject=subject,
        correction=True,
        detailed=True,
    )
    effect = anova.iloc[0]

    epsilon = float(effect["eps"])
    df1_uncorrected = k - 1
    df2_uncorrected = (k - 1) * (n_subjects - 1)

    if apply_gg:
        final_p = float(effect["p_GG_corr"])
        final_df1 = epsilon * df1_uncorrected
        final_df2 = epsilon * df2_uncorrected
        correction = "Greenhouse-Geisser"
    else:
        final_p = float(effect["p_unc"])
        final_df1 = float(df1_uncorrected)
        final_df2 = float(df2_uncorrected)
        correction = "None"

    omnibus = {
        "test": "Repeated-measures ANOVA",
        "statistic_name": "F",
        "statistic": float(effect["F"]),
        "df1": final_df1,
        "df2": final_df2,
        "p": final_p,
        "effect_name": "eta_G_squared",
        "effect": float(effect["ng2"]),
        "epsilon": epsilon,
        "correction": correction,
        "significant": final_p < alpha,
    }

    posthoc = None
    if final_p < alpha:
        posthoc = pg.pairwise_tests(
            data=long_data,
            dv=dv_name,
            within="configuration",
            subject=subject,
            parametric=True,
            padjust="holm",
            effsize="hedges",
            return_desc=True,
        )
        posthoc = posthoc[[
            "A", "B", "mean_A", "mean_B", "T", "dof",
            "p_unc", "p_corr", "p_adjust", "hedges",
        ]].copy()
        posthoc["significant"] = posthoc["p_corr"] < alpha

    return {
        "sphericity": sphericity,
        "anova": anova,
        "omnibus": omnibus,
        "posthoc": posthoc,
    }


def _run_nonparametric_route(
    analysis_data,
    conditions,
    alpha,
):
    n_subjects = len(analysis_data)
    k = len(conditions)

    friedman = friedmanchisquare(
        *[analysis_data[condition].to_numpy() for condition in conditions]
    )
    q_statistic = float(friedman.statistic)
    p_value = float(friedman.pvalue)
    kendalls_w = q_statistic / (n_subjects * (k - 1))

    omnibus = {
        "test": "Friedman",
        "statistic_name": "Q",
        "statistic": q_statistic,
        "df1": float(k - 1),
        "df2": None,
        "p": p_value,
        "effect_name": "Kendalls_W",
        "effect": kendalls_w,
        "epsilon": None,
        "correction": "Not applicable",
        "significant": p_value < alpha,
    }

    posthoc = None
    if p_value < alpha:
        rows = []
        for configuration_a, configuration_b in combinations(conditions, 2):
            values_a = analysis_data[configuration_a].to_numpy()
            values_b = analysis_data[configuration_b].to_numpy()

            if np.allclose(values_a - values_b, 0):
                w_value = 0.0
                pair_p = 1.0
                rbc = 0.0
            else:
                wilcoxon = pg.wilcoxon(
                    values_a,
                    values_b,
                    alternative="two-sided",
                ).iloc[0]
                w_value = float(wilcoxon["W-val"])
                pair_p = float(wilcoxon["p-val"])
                rbc = float(wilcoxon["RBC"])

            rows.append({
                "A": configuration_a,
                "B": configuration_b,
                "mean_A": float(np.mean(values_a)),
                "mean_B": float(np.mean(values_b)),
                "W": w_value,
                "p_unc": pair_p,
                "RBC": rbc,
            })

        posthoc = pd.DataFrame(rows)
        reject, corrected_p = pg.multicomp(
            posthoc["p_unc"].to_numpy(),
            alpha=alpha,
            method="holm",
        )
        posthoc["p_corr"] = corrected_p
        posthoc["significant"] = reject

    return {
        "sphericity": None,
        "anova": None,
        "omnibus": omnibus,
        "posthoc": posthoc,
    }


def _create_pairwise_paper_table(posthoc, route):
    """Build a paper-friendly table with every pairwise comparison.

    The descriptively better configuration is always listed first, so the
    test statistic and effect size are reported as magnitudes: direction is
    already encoded by the comparison order. Unlike `significant_posthoc`,
    this table is not filtered by significance — every pair produced by the
    post-hoc procedure is included, with a `Significant` column flagging
    which comparisons crossed alpha after Holm correction.
    """
    if posthoc is None:
        return None

    rows = []
    for _, row in posthoc.iterrows():
        if row["mean_A"] >= row["mean_B"]:
            better, worse = row["A"], row["B"]
        else:
            better, worse = row["B"], row["A"]

        if route == "parametric":
            statistic_name = "t"
            statistic = abs(float(row["T"]))
            effect_name = "g_av"
            effect = abs(float(row["hedges"]))
        else:
            statistic_name = "W"
            statistic = float(row["W"])
            effect_name = "r_rb"
            effect = abs(float(row["RBC"]))

        rows.append({
            "Comparison": f"{better} vs. {worse}",
            "Better": better,
            "Worse": worse,
            "Statistic name": statistic_name,
            "Statistic": statistic,
            "p_raw": float(row["p_unc"]),
            "p_Holm": float(row["p_corr"]),
            "Effect name": effect_name,
            "Effect": effect,
            "Significant": bool(row["significant"]),
        })

    # Significant comparisons first, then ordered by effect magnitude.
    return (
        pd.DataFrame(rows)
        .sort_values(["Significant", "Effect"], ascending=[False, False])
        .reset_index(drop=True)
    )


def run_automatic_repeated_measures_analysis(
    data,
    subject,
    conditions,
    dv_name="macro_f1",
    alpha=0.05,
):
    """Select and run the prespecified repeated-measures analysis.

    IMPORTANT: all pairwise comparisons are retained and displayed,
    including non-significant ones — not only the significant subset.
    """

    if len(conditions) < 3:
        raise ValueError("At least three configurations are required.")

    analysis_data = data[[subject, *conditions]].dropna().copy()
    long_data = analysis_data.melt(
        id_vars=subject,
        value_vars=conditions,
        var_name="configuration",
        value_name=dv_name,
    )

    descriptive = (
        long_data.groupby("configuration")[dv_name]
        .agg(N="count", Mean="mean", SD="std")
        .sort_values("Mean", ascending=False)
    )

    shapiro_rows = []
    for condition in conditions:
        normality = pg.normality(
            analysis_data[condition],
            method="shapiro",
            alpha=alpha,
        ).iloc[0]
        shapiro_rows.append({
            "Configuration": condition,
            "N": int(analysis_data[condition].notna().sum()),
            "W": float(normality["W"]),
            "p": float(normality["pval"]),
            "Normality not rejected": bool(normality["normal"]),
        })

    shapiro = pd.DataFrame(shapiro_rows).set_index("Configuration")
    all_normal = bool(shapiro["Normality not rejected"].all())

    if all_normal:
        route = "parametric"
        inferential = _run_parametric_route(
            long_data=long_data,
            analysis_data=analysis_data,
            subject=subject,
            conditions=conditions,
            dv_name=dv_name,
            alpha=alpha,
        )
    else:
        route = "non-parametric"
        inferential = _run_nonparametric_route(
            analysis_data=analysis_data,
            conditions=conditions,
            alpha=alpha,
        )

    posthoc = inferential["posthoc"]
    significant_posthoc = None
    pairwise_paper_table = None
    if posthoc is not None:
        significant_posthoc = posthoc.loc[posthoc["significant"]].copy()
        pairwise_paper_table = _create_pairwise_paper_table(posthoc, route)

    print("DESCRIPTIVE STATISTICS")
    display(descriptive.round({"Mean": 4, "SD": 4}))


    print("SHAPIRO-WILK TESTS")

    shapiro_display = shapiro.copy()
    shapiro_display["W"] = shapiro_display["W"].round(4)
    shapiro_display["p"] = shapiro_display["p"].map(
        _format_p_plain
    )

    display(shapiro_display)


    print(f"SELECTED ROUTE: {route}")

    if inferential["sphericity"] is not None:
        sphericity = inferential["sphericity"]

        sphericity_display = pd.DataFrame([{
            "Mauchly W": round(sphericity.W, 4),
            "chi-square": round(sphericity.chi2, 4),
            "df": sphericity.dof,
            "p": _format_p_plain(sphericity.pval),
            "Sphericity": bool(sphericity.spher),
        }])

        display(sphericity_display)


    omnibus_display = pd.DataFrame([
        inferential["omnibus"]
    ]).copy()

    omnibus_display["p"] = omnibus_display["p"].map(
        _format_p_plain
    )

    display(omnibus_display.round({
        "statistic": 4,
        "df1": 4,
        "df2": 4,
        "effect": 4,
        "epsilon": 4,
    }))


    if posthoc is None:
        print(
            "The omnibus test was not significant; "
            "post-hoc comparisons were not performed."
        )

    else:
        print(f"ALL HOLM-CORRECTED PAIRWISE COMPARISONS ({len(posthoc)} total)")

        all_posthoc_display = posthoc.copy()
        all_posthoc_display["p_unc"] = (
            all_posthoc_display["p_unc"].map(_format_p_plain)
        )
        all_posthoc_display["p_corr"] = (
            all_posthoc_display["p_corr"].map(_format_p_plain)
        )

        display(all_posthoc_display.round({
            "mean_A": 4,
            "mean_B": 4,
            "T": 4,
            "dof": 1,
            "hedges": 4,
            "W": 4,
            "RBC": 4,
        }))

        print("PAIRWISE TABLE FOR PAPER")

        paper_display = pairwise_paper_table.copy()
        paper_display["p_raw"] = paper_display["p_raw"].map(_format_p_plain)
        paper_display["p_Holm"] = paper_display["p_Holm"].map(_format_p_plain)

        display(paper_display.round({
            "Statistic": 4,
            "Effect": 4,
        }))

        print(
            "SIGNIFICANT HOLM-CORRECTED COMPARISONS "
            f"({len(significant_posthoc)} of {len(posthoc)})"
        )

        if significant_posthoc.empty:
            print(
                "No pairwise comparison remained significant "
                "after Holm correction."
            )

        else:
            significant_display = significant_posthoc.copy()

            significant_display["p_unc"] = (
                significant_display["p_unc"].map(_format_p_plain)
            )
            significant_display["p_corr"] = (
                significant_display["p_corr"].map(_format_p_plain)
            )

            display(significant_display.round({
                "mean_A": 4,
                "mean_B": 4,
                "T": 4,
                "dof": 1,
                "hedges": 4,
                "W": 4,
                "RBC": 4,
            }))

    return {
        "route": route,
        "analysis_data": analysis_data,
        "long_data": long_data,
        "descriptive": descriptive,
        "shapiro": shapiro,
        "sphericity": inferential["sphericity"],
        "anova": inferential["anova"],
        "omnibus": inferential["omnibus"],
        "posthoc": posthoc,
        "significant_posthoc": significant_posthoc,
        "pairwise_paper_table": pairwise_paper_table,
        "alpha": alpha,
    }


def generate_dynamic_report(
    results,
    report_title,
    labels=None,
    reference=None,
):
    """Generate a Markdown results narrative from the current analysis."""

    labels = labels or {}
    descriptive = results["descriptive"]
    shapiro = results["shapiro"]
    omnibus = results["omnibus"]
    posthoc = results["posthoc"]
    significant = results["significant_posthoc"]
    route = results["route"]

    ranking_lines = []
    for configuration, row in descriptive.iterrows():
        ranking_lines.append(
            f"- **{_pretty_label(configuration, labels)}**: "
            f"mean Macro-F1 = {row['Mean']:.3f}, SD = {row['SD']:.3f}"
        )

    non_normal = shapiro.index[
        ~shapiro["Normality not rejected"]
    ].tolist()

    if route == "parametric":
        normality_text = (
            "Shapiro–Wilk did not reject normality for any configuration; "
            "therefore, the parametric repeated-measures route was selected."
        )
        sphericity = results["sphericity"]
        if sphericity.pval < results["alpha"]:
            sphericity_text = (
                "Mauchly's test indicated a violation of sphericity "
                f"($W={_format_decimal(sphericity.W)}$, "
                f"{_format_p(sphericity.pval)}); the Greenhouse–Geisser "
                "correction was used."
            )
        else:
            sphericity_text = (
                "Mauchly's test did not indicate a violation of sphericity "
                f"($W={_format_decimal(sphericity.W)}$, "
                f"{_format_p(sphericity.pval)})."
            )

        omnibus_text = (
            "The repeated-measures ANOVA "
            + ("detected" if omnibus["significant"] else "did not detect")
            + " an overall configuration effect, "
            + f"$F({omnibus['df1']:.2f},{omnibus['df2']:.2f})="
            + f"{omnibus['statistic']:.2f}$, {_format_p(omnibus['p'])}, "
            + f"$\\eta_G^2={_format_decimal(omnibus['effect'])}$."
        )
        effect_column = "hedges"
        effect_label = "Hedges' $g$"
        statistic_column = "T"
        statistic_label = "$t$"
    else:
        normal_names = ", ".join(
            _pretty_label(configuration, labels)
            for configuration in non_normal
        )
        normality_text = (
            "Shapiro–Wilk rejected normality for at least one configuration "
            f"({normal_names}); therefore, the non-parametric route was selected."
        )
        sphericity_text = "Mauchly's test is not required for the Friedman route."
        omnibus_text = (
            "The Friedman test "
            + ("detected" if omnibus["significant"] else "did not detect")
            + " an overall configuration effect, "
            + f"$\\chi^2({int(omnibus['df1'])})={omnibus['statistic']:.2f}$, "
            + f"{_format_p(omnibus['p'])}, "
            + f"Kendall's $W={_format_decimal(omnibus['effect'])}$."
        )
        effect_column = "RBC"
        effect_label = "$r_{rb}$"
        statistic_column = "W"
        statistic_label = "$W$"

    pairwise_lines = []
    if posthoc is None:
        pairwise_text = (
            "Because the omnibus test was not significant, confirmatory "
            "pairwise post-hoc tests were not performed."
        )
    elif significant is None or significant.empty:
        pairwise_text = (
            "No pairwise comparison remained significant after Holm correction."
        )
    else:
        for _, row in significant.iterrows():
            if row["mean_A"] >= row["mean_B"]:
                better, worse = row["A"], row["B"]
            else:
                better, worse = row["B"], row["A"]

            pairwise_lines.append(
                f"- **{_pretty_label(better, labels)}** outperformed "
                f"**{_pretty_label(worse, labels)}** "
                f"({statistic_label} = {row[statistic_column]:.2f}, "
                f"{_format_p(row['p_corr'])}, "
                f"{effect_label} = {abs(row[effect_column]):.3f})"
            )
        pairwise_text = (
            "The following comparisons remained significant after "
            "Holm correction:\n\n" + "\n".join(pairwise_lines)
        )

    reference_text = ""
    if reference and reference in descriptive.index:
        reference_label = _pretty_label(reference, labels)
        if posthoc is None:
            reference_text = (
                f"No confirmatory pairwise claim relative to **{reference_label}** "
                "is made because the omnibus test was not significant."
            )
        else:
            improved = []
            worsened = []
            for _, row in significant.iterrows():
                if reference not in {row["A"], row["B"]}:
                    continue
                other = row["B"] if row["A"] == reference else row["A"]
                if descriptive.loc[other, "Mean"] > descriptive.loc[reference, "Mean"]:
                    improved.append(_pretty_label(other, labels))
                else:
                    worsened.append(_pretty_label(other, labels))

            parts = []
            if improved:
                parts.append(
                    "Significantly higher than the reference: "
                    + ", ".join(improved)
                    + "."
                )
            if worsened:
                parts.append(
                    "Significantly lower than the reference: "
                    + ", ".join(worsened)
                    + "."
                )
            if not parts:
                parts.append(
                    f"No configuration differed significantly from "
                    f"**{reference_label}** after Holm correction."
                )
            reference_text = " ".join(parts)

    best_configuration = descriptive.index[0]
    best_label = _pretty_label(best_configuration, labels)
    if omnibus["significant"]:
        conclusion = (
            f"**{best_label}** had the highest descriptive mean. The omnibus "
            "test detected differences among configurations; statistical "
            "claims about specific pairs are limited to the corrected "
            "comparisons reported above."
        )
    else:
        conclusion = (
            f"**{best_label}** had the highest descriptive mean, but the "
            "omnibus test did not detect an overall difference. The observed "
            "ordering should therefore be interpreted descriptively."
        )

    report = f"""
## {report_title}

### Descriptive results

{chr(10).join(ranking_lines)}

### Assumptions and omnibus test

{normality_text} {sphericity_text} {omnibus_text}

### Pairwise evidence

{pairwise_text}

{reference_text}

### Data-driven conclusion

{conclusion}
"""
    return report



## Execute the analysis and generate the narrative

The results object and the Markdown interpretation below are created from the data loaded in the current runtime. No numerical result is stored as static notebook text.


In [ ]:
analysis_results = run_automatic_repeated_measures_analysis(
    data=df,
    subject=SUBJECT,
    conditions=CONFIGURATIONS,
)

generated_report = generate_dynamic_report(
    results=analysis_results,
    report_title="RQ3.1 interpretation generated from the current data",
    labels=LABELS,
    reference=REFERENCE_CONFIGURATION,
)

display(Markdown(generated_report))


## Reproducibility notes

- The numerical narrative above is generated only after executing the current data.
- The analysis uses the four prespecified RQ3.1 input representations: message-only, diff-only, message+diff, and git-show.
- Complete-case filtering is applied across all four representations, and removed project identifiers are printed.
- All statistical decisions use unrounded values; rounding is applied only for presentation.
- Review the generated scientific wording before copying it into the paper; automation prevents stale numbers but does not replace substantive interpretation.